# Task 5 — Step 2: Tavily Web Search

Runs one Tavily search per sampled entity (from Step 1) and saves the top results
(title, url, content snippet) for Step 3 (LLM extraction + labeling).

Resumable: re-running this notebook skips entities that already have results in
`tavily_results.jsonl`.

Set `DATASET` below and run once per dataset.

In [8]:
DATASET = "wdc-products"   # "dblp-scholar" | "wdc-products"
MAX_RESULTS = 5
SEARCH_DEPTH = "basic"
MAX_QUERY_LEN = 400        # truncate overly long queries (noisy DBLP venue strings)
SLEEP_BETWEEN = 0.3        # seconds between requests
MAX_RETRIES = 3

In [9]:
import json
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tavily import TavilyClient

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / "data" / "processed" / DATASET
QUERIES_PATH = PROCESSED / "web_query_entities.csv"
OUTPUT = PROCESSED / "tavily_results.jsonl"

load_dotenv(ROOT / ".env")
client = TavilyClient()  # reads TAVILY_API_KEY from env

print(f"Dataset : {DATASET}")
print(f"Queries : {QUERIES_PATH}")
print(f"Output  : {OUTPUT}")

Dataset : wdc-products
Queries : /Users/abd/Developer/thesis-project/data/processed/wdc-products/web_query_entities.csv
Output  : /Users/abd/Developer/thesis-project/data/processed/wdc-products/tavily_results.jsonl


In [10]:
entities = pd.read_csv(QUERIES_PATH)
print(f"Entities to search: {len(entities)}")

already_done = set()
n_existing = 0
if OUTPUT.exists():
    with open(OUTPUT) as f:
        for line in f:
            rec = json.loads(line)
            already_done.add((rec["id"], rec["side"]))
            n_existing += 1
    print(f"Resuming: {n_existing} entities already searched")
else:
    print("Starting fresh — no existing results file found")

remaining = entities[~entities.apply(lambda r: (r["id"], r["side"]) in already_done, axis=1)]
print(f"Remaining: {len(remaining)} entities")

Entities to search: 300
Starting fresh — no existing results file found
Remaining: 300 entities


In [11]:
def search_with_retries(query: str) -> dict | None:
    for attempt in range(MAX_RETRIES):
        try:
            return client.search(query, search_depth=SEARCH_DEPTH, max_results=MAX_RESULTS)
        except Exception as e:
            wait = 2 ** attempt
            print(f"    [retry {attempt+1}/{MAX_RETRIES}] {type(e).__name__}: {e} — waiting {wait}s")
            time.sleep(wait)
    print("    [skip] all retries failed")
    return None

In [12]:
n_done = 0
n_empty = 0
n_failed = 0

with open(OUTPUT, "a") as out_f:
    for i, row in remaining.iterrows():
        query = str(row["query"])[:MAX_QUERY_LEN]
        if not query.strip():
            continue

        response = search_with_retries(query)
        if response is None:
            n_failed += 1
            continue

        results = [
            {"title": r.get("title", ""), "url": r.get("url", ""), "content": r.get("content", "")}
            for r in response.get("results", [])
        ]
        if not results:
            n_empty += 1

        record = {
            "id": int(row["id"]),
            "side": row["side"],
            "bucket": row["bucket"],
            "text": row["text"],
            "query": query,
            "results": results,
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()
        n_done += 1

        if n_done % 25 == 0:
            print(f"  {n_done}/{len(remaining)} searched...")

        time.sleep(SLEEP_BETWEEN)

print(f"\nDone. Searched {n_done} entities ({n_empty} returned 0 results, {n_failed} failed).")

  25/300 searched...
  50/300 searched...
  75/300 searched...
  100/300 searched...
  125/300 searched...
  150/300 searched...
  175/300 searched...
  200/300 searched...
  225/300 searched...
  250/300 searched...
  275/300 searched...
  300/300 searched...

Done. Searched 300 entities (0 returned 0 results, 0 failed).


In [13]:
# Summary over the full output file
all_records = [json.loads(l) for l in open(OUTPUT)]
n_results = [len(r["results"]) for r in all_records]

print(f"Total entities searched : {len(all_records)}")
print(f"Avg results per query   : {sum(n_results)/len(n_results):.2f}")
print(f"Entities with 0 results : {sum(1 for n in n_results if n == 0)}")
print(f"Entities with >=3 results: {sum(1 for n in n_results if n >= 3)}")

Total entities searched : 300
Avg results per query   : 5.00
Entities with 0 results : 0
Entities with >=3 results: 300


In [14]:
# Preview a few examples
for rec in all_records[:3]:
    print(f"\n{'='*70}")
    print(f"[{rec['side']}] id={rec['id']}  bucket={rec['bucket']!r}")
    print(f"query: {rec['query']!r}")
    for r in rec["results"][:2]:
        print(f"  - {r['title']!r}")
        print(f"    {r['url']}")
        print(f"    {r['content'][:150]}...")


[left] id=49605449  bucket='Brother'
query: 'Brother Brother HL-L6300DW Business Laser Printer for Mid-Size Workgroups -HL-L6300DW'
  - 'Laser Printer Sales Menomonee Falls Wisconsin | Get A Quote!'
    https://wisconsin-copiers.com/wisconsin/menomonee-falls/laser-printer-sales.php
    $399.00 + TAX. BROTHER HLL6300DW. Business Laser Printer for Mid-Size Workgroups with Higher Print Volumes. ​Includes high-yield toner cartridge; Adva...
  - '(763) 509-0054 - laser printer sales rentals & leasing minnesota'
    https://promtlocaltech.com/laser-printer-sales
    LASER PRINTER SALES RENTALS & LEASING MINNESOTA ; $399.00 + TAX. BROTHER HLL6300DW. Business Laser Printer for Mid-Size Workgroups with Higher Print V...

[left] id=7873500  bucket='TP-LINK'
query: 'TP-LINK Access Point TP-LINK Wireless N 300Mbs B/G/N - 3 Antenas'
  - 'TL-WA801N | 300Mbps Wireless N Access Point | TP-Link'
    https://www.tp-link.com/us/home-networking/access-point/tl-wa801n
    Please turn it on for the best ex